 ## importing libraries and loading cleaned data

In [1]:
## importing libraries and loading cleaned data
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

FILE_DIR = Path.cwd() / "Data_Collection"
DATA_PATH = FILE_DIR / "country_crop_data.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Cleaned data was not found. Run 01_Data_Preprocessing.ipynb first."
    )

country_crop_data = pd.read_csv(DATA_PATH)
country_crop_data["Year"] = country_crop_data["Year"].astype(int)

required_cols = {"Area Code", "Item Code", "Area", "Item", "Year", "Yield_kg_ha"}
missing_cols = required_cols - set(country_crop_data.columns)
if missing_cols:
    raise ValueError(f"Required columns are missing: {sorted(missing_cols)}")

display(country_crop_data.head())
print("Rows:", len(country_crop_data), "| Year range:",
      country_crop_data["Year"].min(), "-", country_crop_data["Year"].max())

,Area Code,Area,Item Code,Item,Element,Unit,Year,Yield_kg_ha
0,1,Armenia,15,Wheat,Yield,kg/ha,1992,2160.0
1,1,Armenia,15,Wheat,Yield,kg/ha,1993,2225.7
2,1,Armenia,15,Wheat,Yield,kg/ha,1994,1784.2
3,1,Armenia,15,Wheat,Yield,kg/ha,1995,2548.0
4,1,Armenia,15,Wheat,Yield,kg/ha,1996,2356.9


Rows: 24588 | Year range: 1961 - 2023


 ## Engineering exact calendar-year lag features

In [2]:
## Engineering exact calendar-year lag features
# Joining on shifted Year avoids treating a multi-year gap as if it were one year.
model_data = country_crop_data.copy()
keys = ["Area Code", "Item Code", "Year"]

for lag in (1, 2, 3):
    previous = country_crop_data[keys + ["Yield_kg_ha"]].copy()
    previous["Year"] = previous["Year"] + lag
    previous = previous.rename(columns={"Yield_kg_ha": f"Yield_lag_{lag}"})
    model_data = model_data.merge(
        previous, on=keys, how="left", validate="one_to_one"
    )

lag_cols = ["Yield_lag_1", "Yield_lag_2", "Yield_lag_3"]
model_data["Yield_rolling_mean_3"] = model_data[lag_cols].mean(axis=1)
model_data.loc[model_data[lag_cols].isna().any(axis=1), "Yield_rolling_mean_3"] = np.nan

display(model_data[["Area", "Item", "Year", "Yield_kg_ha"] + lag_cols + ["Yield_rolling_mean_3"]].head(12))


,Area,Item,Year,Yield_kg_ha,Yield_lag_1,Yield_lag_2,Yield_lag_3,Yield_rolling_mean_3
0,Armenia,Wheat,1992,2160.0,NaN,NaN,NaN,NaN
1,Armenia,Wheat,1993,2225.7,2160.0,NaN,NaN,NaN
2,Armenia,Wheat,1994,1784.2,2225.7,2160.0,NaN,NaN
3,Armenia,Wheat,1995,2548.0,1784.2,2225.7,2160.0,2056.633333
4,Armenia,Wheat,1996,2356.9,2548.0,1784.2,2225.7,2185.966667
5,Armenia,Wheat,1997,1700.9,2356.9,2548.0,1784.2,2229.700000
6,Armenia,Wheat,1998,2065.1,1700.9,2356.9,2548.0,2201.933333
7,Armenia,Wheat,1999,1946.6,2065.1,1700.9,2356.9,2040.966667
8,Armenia,Wheat,2000,1667.9,1946.6,2065.1,1700.9,1904.200000
9,Armenia,Wheat,2001,2226.3,1667.9,1946.6,2065.1,1893.200000


 ## defining model features and target

In [3]:
## defining model features and target
numeric_features = ["Yield_lag_1", "Yield_lag_2", "Yield_lag_3", "Yield_rolling_mean_3"]
categorical_features = ["Area", "Item"]
feature_cols = numeric_features + categorical_features
target_col = "Yield_kg_ha"

eligible_data = model_data.dropna(subset=numeric_features + [target_col]).copy()
if eligible_data.empty:
    raise ValueError("No rows have complete lag features. Check the source years and preprocessing.")
print("Eligible rows:", len(eligible_data))
print("Eligible year range:", eligible_data["Year"].min(), "-", eligible_data["Year"].max())
print("Missing model features:", int(eligible_data[feature_cols].isna().sum().sum()))


Eligible rows: 23189
Eligible year range: 1964 - 2023
Missing model features: 0


 ## splitting data chronologically into train, validation, and test periods

In [4]:
## splitting data chronologically into train, validation, and test periods
# Train: through 2013; validation: 2014-2018; test: 2019-2023.
train_df = eligible_data.loc[eligible_data["Year"] <= 2013].copy()
val_df = eligible_data.loc[eligible_data["Year"].between(2014, 2018)].copy()
test_df = eligible_data.loc[eligible_data["Year"].between(2019, 2023)].copy()

if min(len(train_df), len(val_df), len(test_df)) == 0:
    raise ValueError("One split is empty. Check whether the dataset contains the required years.")

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_val, y_val = val_df[feature_cols], val_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

print("Train:", len(train_df), train_df["Year"].min(), "-", train_df["Year"].max())
print("Validation:", len(val_df), val_df["Year"].min(), "-", val_df["Year"].max())
print("Test:", len(test_df), test_df["Year"].min(), "-", test_df["Year"].max())


Train: 19036 1964 - 2013
Validation: 2069 2014 - 2018
Test: 2084 2019 - 2023


 ## defining evaluation metrics and the previous-year baseline

In [5]:
## defining evaluation metrics and the previous-year baseline
def evaluate_regression(name, actual, predicted):
    return {
        "Model": name,
        "MAE_kg_ha": mean_absolute_error(actual, predicted),
        "RMSE_kg_ha": float(np.sqrt(mean_squared_error(actual, predicted))),
        "R2": r2_score(actual, predicted)
    }

validation_results = []

# Baseline: predict this year's yield using the previous calendar year's yield.
baseline_val_pred = X_val["Yield_lag_1"].to_numpy()
validation_results.append(
    evaluate_regression("Previous-Year Baseline", y_val, baseline_val_pred)
)

print("Validation metrics use 2014–2018 only. The 2019–2023 test set remains untouched.")

Validation metrics use 2014–2018 only. The 2019–2023 test set remains untouched.


 ## comparing linear regression, ridge regression, decision tree, and random forest

In [6]:
## comparing linear regression, ridge regression, decision tree, and random forest
# Numeric-only candidates use the same lag features and chronological split.
# Scaling is fitted on training data only for the linear models.
linear_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

linear_model = Pipeline(steps=[
    ("preprocessor", linear_preprocessor),
    ("model", LinearRegression())
])
linear_model.fit(X_train[numeric_features], y_train)
linear_val_pred = linear_model.predict(X_val[numeric_features])
validation_results.append(
    evaluate_regression("Linear Regression (lags only)", y_val, linear_val_pred)
)

ridge_model = Pipeline(steps=[
    ("preprocessor", linear_preprocessor),
    ("model", Ridge(alpha=1.0))
])
ridge_model.fit(X_train[numeric_features], y_train)
ridge_val_pred = ridge_model.predict(X_val[numeric_features])
validation_results.append(
    evaluate_regression("Ridge Regression (lags only)", y_val, ridge_val_pred)
)

tree_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", DecisionTreeRegressor(
        max_depth=12, min_samples_leaf=5, random_state=42
    ))
])
tree_model.fit(X_train[numeric_features], y_train)
tree_val_pred = tree_model.predict(X_val[numeric_features])
validation_results.append(
    evaluate_regression("Decision Tree (lags only)", y_val, tree_val_pred)
)

rf_lags = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=200, max_depth=12, min_samples_leaf=3,
        random_state=42, n_jobs=-1
    ))
])
rf_lags.fit(X_train[numeric_features], y_train)
rf_lags_val_pred = rf_lags.predict(X_val[numeric_features])
validation_results.append(
    evaluate_regression("Random Forest (lags only)", y_val, rf_lags_val_pred)
)

 ## comparing random forest and gradient boosting with area and crop features

In [7]:
## comparing random forest and gradient boosting with area and crop features
# OneHotEncoder is fitted on training data only. Unknown categories are handled safely.
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200, max_depth=12, min_samples_leaf=3,
        random_state=42, n_jobs=-1
    ))
])
rf_pipeline.fit(X_train, y_train)
rf_val_pred = rf_pipeline.predict(X_val)
validation_results.append(
    evaluate_regression("Random Forest + Area + Crop", y_val, rf_val_pred)
)

gb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(
        n_estimators=150, learning_rate=0.05, max_depth=3,
        loss="squared_error", random_state=42
    ))
])
gb_pipeline.fit(X_train, y_train)
gb_val_pred = gb_pipeline.predict(X_val)
validation_results.append(
    evaluate_regression("Gradient Boosting + Area + Crop", y_val, gb_val_pred)
)

 ## optionally comparing XGBoost on the validation set

In [8]:
## optionally comparing XGBoost on the validation set
# XGBoost is optional. If it is not installed, the rest of the notebook still runs.
# This candidate uses the same numeric lag features as the numeric-only tree models.
try:
    from xgboost import XGBRegressor

    xgb_model = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBRegressor(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=1.0,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        ))
    ])
    xgb_model.fit(X_train[numeric_features], y_train)
    xgb_val_pred = xgb_model.predict(X_val[numeric_features])
    validation_results.append(
        evaluate_regression("XGBoost (lags only)", y_val, xgb_val_pred)
    )
except ImportError:
    print("XGBoost is not installed; skipping it. Optional install: %pip install xgboost")

print("Models compared:", len(validation_results))

Models compared: 8


 ## comparing validation results and documenting the model-selection decision

In [9]:
## comparing validation results and documenting the model-selection decision
validation_results_df = (
    pd.DataFrame(validation_results)
    .sort_values(["MAE_kg_ha", "RMSE_kg_ha"])
    .reset_index(drop=True)
)
display(validation_results_df.round(3))

# Select the best candidate by validation MAE only; use RMSE as a tie-breaker.
validation_winner = validation_results_df.iloc[0]["Model"]
print("Best validation candidate by MAE:", validation_winner)


,Model,MAE_kg_ha,RMSE_kg_ha,R2
0,Ridge Regression (lags only),439.694,1068.246,0.898
1,Linear Regression (lags only),439.702,1068.321,0.898
2,Gradient Boosting + Area + Crop,445.286,1047.842,0.902
3,Random Forest + Area + Crop,447.107,1002.191,0.910
4,Random Forest (lags only),449.371,1005.605,0.910
5,XGBoost (lags only),464.071,1062.548,0.899
6,Previous-Year Baseline,466.977,1237.548,0.863
7,Decision Tree (lags only),499.245,1057.190,0.900


Best validation candidate by MAE: Ridge Regression (lags only)


 ## saving feature definitions and split datasets for the final notebook

In [12]:

## saving feature definitions and split datasets for the final notebook

import json

FILE_DIR.mkdir(parents=True, exist_ok=True)

# Use the validation winner as the deployment candidate
selected_model_name = validation_winner

# Save eligible modeling data
eligible_path = FILE_DIR / "eligible_model_data.csv"
eligible_data.to_csv(eligible_path, index=False)

# Save model comparison results
comparison_path = FILE_DIR / "validation_model_comparison.csv"
validation_results_df.to_csv(comparison_path, index=False)

# Save feature definitions and model-selection metadata
feature_metadata = {
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "target": target_col,
    "train_year_max": 2013,
    "validation_years": [2014, 2018],
    "test_years_reserved": [2019, 2023],
    "deployment_candidate": selected_model_name,
    "validation_winner": validation_winner
}

metadata_path = FILE_DIR / "feature_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(feature_metadata, f, indent=2)

print("Saved:", eligible_path)
print("Saved:", comparison_path)
print("Saved:", metadata_path)
print("Selected validation winner:", selected_model_name)
print("Next: run 04_Final_Model.ipynb")


Saved: C:\Users\acer\capstone_Project\Agriculture_Yield_Prediction\Crop_Field_Prediction_Using_ML\Data_Collection\eligible_model_data.csv
Saved: C:\Users\acer\capstone_Project\Agriculture_Yield_Prediction\Crop_Field_Prediction_Using_ML\Data_Collection\validation_model_comparison.csv
Saved: C:\Users\acer\capstone_Project\Agriculture_Yield_Prediction\Crop_Field_Prediction_Using_ML\Data_Collection\feature_metadata.json
Selected validation winner: Ridge Regression (lags only)
Next: run 04_Final_Model.ipynb
